In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [2]:
spark = SparkSession\
    .builder\
    .master("local[*]")\
    .appName("Files")\
    .config("spark.log.level", "WARN")\
    .config("spark.jars", "/opt/avro/spark-avro_2.12-3.5.8.jar")\
    .config("spark.sql.debug.maxToStringFields", 1024)\
    .getOrCreate()

26/06/27 18:40:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".


In [3]:
print("spark.version == ", spark.version)

spark.version ==  3.5.8


## CSV

### Пример 1

In [ ]:
!cat data/people.csv

In [ ]:
path = "data/people.csv"
df1 = spark.read.csv(path)
df1.show()

In [ ]:
df2 = spark.read.option("delimiter", ";").csv(path)
df2.show()

In [ ]:
df3 = spark.read.option("delimiter", ";").option("header", "true").csv(path)
df3.show()

In [ ]:
df4 = spark.read.options(delimiter=";", header=True).csv(path)
df4.show()

### Пример 2

In [ ]:
!head data/books.csv

In [ ]:
booksPath = "data/books.csv"
books = spark.read.option("header", "true").csv("data/books.csv")
books.show(5, False)

In [ ]:
books.printSchema()

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

booksSchema = StructType([
    StructField("Name", StringType(), nullable = True),
    StructField("Author", StringType(), nullable = True),
    StructField("User Rating", DoubleType(), nullable = True),
    StructField("Reviews", IntegerType(), nullable = True),
    StructField("Price", IntegerType(), nullable = True),
    StructField("Year", IntegerType(), nullable = True),
    StructField("Genre", StringType(), nullable = True)
])

In [ ]:
booksSchemaDF = spark.read.option("header", "true").schema(booksSchema).csv("data/books.csv")
booksSchemaDF.printSchema()

In [ ]:
booksSchemaDF.show(5, False)

In [ ]:
booksInferDF = spark.read.option("header", "true").option("inferSchema ", "true").csv("data/books.csv")
booksInferDF.printSchema()

## JSON

### Пример 1

In [ ]:
!head -5 data/customer.json

In [ ]:
!head -1 data/customer.json | jq

In [ ]:
customer = spark.read.json("data/customer.json")
customer.printSchema()

In [ ]:
customer.show(5, False)

### Пример 2

In [ ]:
!head -20 data/countries.json

In [ ]:
countries = spark\
    .read\
    .format("json")\
    .option("multiLine", "true")\
    .load("data/countries.json")

In [ ]:
countries.show(5, False)

In [ ]:
countries.printSchema()

## Parquet

In [ ]:
!pip install parquet-tools

In [ ]:
data1 = list(map(lambda x: [x, x * x], [1, 2, 3, 4, 5]))
squaresDF = spark.createDataFrame(data1).withColumnsRenamed({"_1": "value", "_2": "square"})
squaresDF.show()

In [ ]:
squaresDF.write.mode("overwrite").parquet("data/test_table/key=1")

In [ ]:
data2 = list(map(lambda x: [x, x * x * x], [6, 7, 8, 9, 10]))
cubesDF = spark.createDataFrame(data2).withColumnsRenamed({"_1": "value", "_2": "cube"})
cubesDF.show()

In [ ]:
cubesDF.write.mode("overwrite").parquet("data/test_table/key=2")

In [ ]:
mergedDF = spark.read.option("mergeSchema", "true").parquet("data/test_table")
mergedDF.printSchema()

In [ ]:
mergedDF.show()

In [ ]:
mergedDF.write.mode("overwrite").parquet("data/merged.parquet")

In [ ]:
unMergedDF = spark.read.option("mergeSchema", "false").parquet("data/test_table")
unMergedDF.printSchema()

In [ ]:
unMergedDF.show()

In [ ]:
!parquet-tools show data/merged.parquet

In [ ]:
!ls -1 data/merged.parquet | grep -v _SUCCESS

In [ ]:
merged = %system ls -1 data/merged.parquet
file1 = merged[0]

In [ ]:
print(file1)

In [ ]:
!parquet-tools inspect data/merged.parquet/{file1}

In [ ]:
!ls -1 data/test_table/key=2

In [ ]:
key2Files = %system ls -1 data/test_table/key=2
file2 = key2Files[1]

In [ ]:
!parquet-tools inspect data/test_table/key=2/{file2}

## ORC

### Пример 1

In [ ]:
users = spark.read.orc("data/users.orc")
users.printSchema()

In [ ]:
users.show()

### Пример 2

In [ ]:
example = spark.read.orc("data/example.orc")
example.printSchema()

In [ ]:
example.show()

## Avro

In [4]:
usersDF = spark.read.format("avro").load("data/users.avro")
usersDF.printSchema()

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)
 |-- favorite_numbers: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [5]:
usersDF.show()

+------+--------------+----------------+
|  name|favorite_color|favorite_numbers|
+------+--------------+----------------+
|Alyssa|          NULL|  [3, 9, 15, 20]|
|   Ben|           red|              []|
+------+--------------+----------------+



In [7]:
usersDF\
    .select("name", "favorite_color")\
    .write\
    .mode("overwrite")\
    .format("avro")\
    .save("data/namesAndFavColors.avro")

In [8]:
namesAndFavColorsDF = spark.read.format("avro").load("data/namesAndFavColors.avro")
namesAndFavColorsDF.printSchema()

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)



In [9]:
namesAndFavColorsDF.show()

+------+--------------+
|  name|favorite_color|
+------+--------------+
|Alyssa|          NULL|
|   Ben|           red|
+------+--------------+



In [10]:
!java -jar /opt/avro/avro-tools-1.12.1.jar compile schema data/user.avsc data

Input files to compile:
  data/user.avsc


In [11]:
!head data/example/avro/User.java

/*
 * Autogenerated by Avro
 *
 * DO NOT EDIT DIRECTLY
 */
package example.avro;

import org.apache.avro.specific.SpecificData;
import org.apache.avro.util.Utf8;
import org.apache.avro.message.BinaryMessageEncoder;
